# Accessing the Digital Earth Normalised Radar Backscatter Product for Sentinel-1 (Collection 0) Using the STAC API

This notebook shows a number of additional features that can be taken advantage of when loading through Geoscience Australia's STAC API.

## Important STAC terminology
The STAC standard has four important components:

* **Catalog**: A structure for organising multiple datasets managed by a given provider.
* **Collection**: A structure for organising all items in a single dataset.
* **Item** A single spatio-temporal item, such as one observation in a dataset.
* **Asset** A single data measurement associated with an item, such as a single band.

For the Normalised Radar Backscatter data, it is important to note that a single *item* corresponds to a single *burst* from a larger acquisition.

## Set-up

### Import required libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from odc.geo import BoundingBox
import odc.geo.xr
import pathlib
import xarray as xr
from pystac_client import Client
from odc.stac import load, configure_s3_access

### Environment set up

In [ ]:
catalog = "https://explorer.dev.dea.ga.gov.au/stac"
stac_client = Client.open(catalog)
configure_s3_access(cloud_defaults=True, aws_unsigned=True)

## Interrogating the catalog and collections

### View available collections

In [ ]:
results = stac_client.collection_search(q="ga_s1_nrb_iw")

collections = [result.id for result in results.collection_list()]

print(f"Available collections: {collections}")

### View collection metadata

Displaying the result from `get_collection()` will provide an expandable JSON output showing the collection-level metadata. 

In [ ]:
collection = stac_client.get_collection("ga_s1_nrb_iw_vv_vh_0")

collection

### View dataset metadata

Many additional metadata properties are available on a dataset level. For this example, we show the metadata associated with a specific item, [ga_s1a_nrb_0-1-0_T002-003369-IW3_20250416T203421Z](https://explorer.dev.dea.ga.gov.au/products/ga_s1_nrb_iw_vv_vh_0/datasets/222a14f0-6fef-5455-8198-1e5e6e2a2d41), which has a dataset id of `222a14f0-6fef-5455-8198-1e5e6e2a2d41`

Displaying the first result from the `item_collection` will provide an expandable JSON output. 
The additional metadata properties and their values can be found under `properties`.

In [ ]:
single_item = stac_client.search(
    collections=["ga_s1_nrb_iw_vv_vh_0"],
    ids=["222a14f0-6fef-5455-8198-1e5e6e2a2d41"]

).item_collection()

single_item[0]

## Basic search and load
When working with the STAC API, there are two steps: 
1. Searching the collection for items that match a query using `pystac-client`
2. Loading the discovered items into an xarray with `odc-stac`

The following shows a basic search and load

### Set search parameters

These are primarily used by the search step, but may also be re-used in load

In [ ]:
# Collections to query
collections_to_search = ["ga_s1_nrb_iw_vv_vh_0"]

# Area of interest
aoi_bbox = BoundingBox(
    left=147.0,
    bottom=-42.5,
    right=148.0,
    top=-41.5,
    crs="EPSG:4326"
)

# Date range of interest
start_date = "2024-06-01"
end_date = "2025-06-30"

aoi_bbox.explore()

### Run search

Note that the search function takes:
* `collections`: a list of collections.
* `datetime`: a date range in the format `"{start}/{end}"`.
* `intersects`: a geometry to intersect.

See the [API](https://pystac-client.readthedocs.io/en/latest/api.html#pystac_client.Client.search) for more information.

In [ ]:
items = stac_client.search(
    collections=collections_to_search,
    datetime=f"{start_date}/{end_date}",
    intersects=aoi_bbox.boundary(),
).item_collection()

print(f"Found {len(items)} items")

### View item-level metadata

Before loading, it can be valuable to understand the general metadata properties of the returned items.
For example, are your items primarily captured in descending or ascending passes? 

In [ ]:
ascending_items = [item for item in items if item.properties["sat:orbit_state"]=="ascending"]
print(f"Number of items with ascending orbit state: {len(ascending_items)}")

descending_items = [item for item in items if item.properties["sat:orbit_state"]=="descending"]
print(f"Number of items with descending orbit state: {len(descending_items)}")

Alternatively, you could look at the number of unique absolute orbits (tracks) or scenes across your items.

In [ ]:
unique_relative_orbits = set([item.properties["sat:relative_orbit"] for item in items])
print(f"The identified items come from {len(unique_relative_orbits)} unique relative orbits")
print(f"The unique relative orbit values are: {unique_relative_orbits}\n")

unique_absolute_orbits = set([item.properties["sat:absolute_orbit"] for item in items])
print(f"The identified items come from {len(unique_absolute_orbits)} unique absolute orbits")
print(f"The unique absolute orbit values are: {unique_absolute_orbits}\n")

unique_scene_ids = set([item.properties["sarard:scene_id"] for item in items])
print(f"The identified items come from {len(unique_scene_ids)} unique scenes")
print(f"The unique scene ids are: {unique_scene_ids}")


### Set load parameters

These parameters specify how the data should be loaded, including which assets (bands) to load, the desired coordinate reference system and resolution, and how data should be grouped (discussed more later in this notebook). The recommended default grouping operation is `"solar_day"`, which combines all observations captured on a single day into one time-step in the loaded xarray.

For this example, we're loading a large area around Canberra, so we set the output resolution to a larger number of metres to avoid consuming too much memory when loading all bursts.

In [ ]:
# Assets to load
assets_to_load = ["VV_gamma0", "VH_gamma0", "mask"]

# CRS and resolution
output_crs = "utm"
output_res = 400 # 20m is the native resolution of the data. Here, we resample to 400m to save memory

# Property or function to group by. "solar_day" is already built into odc-stac
groupy_by_operation = "solar_day"

### Run load

Note that the load function takes:
* `items`: a pystac-client items_collection
* `bands`: a list of assets to load
* `intersects`: a geometry to intersect
* `crs`: a coordinate reference system for the output
* `resolution`: a resolution for the output
* `groupby`: a method or property to group by
* `chunks`: (optionally) a dictionary describing how to chunk the data for lazy loading (recommended)

See the [API](https://odc-stac.readthedocs.io/en/latest/_api/odc.stac.load.html) for more information.

In [ ]:
ds = load(
    items=items,
    bands=assets_to_load,
    intersects=aoi_bbox.boundary(),
    crs=output_crs,
    resolution=output_res,
    groupby=groupy_by_operation,
    chunks={},
)

### View the lazy-loaded xarray

In [ ]:
ds

### Compute the lazy loaded xarray
To save time and memory for this demonstration, we'll only compute the first three time-steps.

In [ ]:
ds = ds.isel(time=range(0,3)).compute()

ds

### Visualise

This step shows how to visualise the first three time-steps from the xarray.
After grouping by solar day, you can see how the Sentinel-1 orbit tracks influence where data is and isn't captured on a given day.

In [ ]:
ds.isel(time=range(0,3)).VV_gamma0.plot.imshow(col="time", col_wrap=3, robust=True, cmap="Greys_r")

## Advanced approaches

### Filtering on metadata values

It is possible to filter on the various metadata properties as part of the search step, which can be valuable for ensuring consistency of data.
For example, with Sentinel-1, you may wish to filter your datasets to only load those observed on descending or ascending paths.

Pystac-client uses the [Common Query Language](https://docs.ogc.org/is/21-065r2/21-065r2.html) standard to search on metadata fields. 
In Python, it's a dictionary with two keys:
* `"op"` which sets the operation to conduct, e.g. `"="`, `"<="`, `"and"`, `"not"`
* `"args"` which is a list containing the property and the value to apply the operation to

The pystac-client documentation has some [examples](https://pystac-client.readthedocs.io/en/stable/tutorials/cql2-filter.html#CQL2-Filters).

Below, we show how to filter items for a single known `scene_id`:

In [ ]:
scene_F7D4_filter = {
    "op": "=",
    "args": [{"property": "sarard:scene_id"}, "S1A_IW_SLC__1SDV_20240921T192428_20240921T192455_055767_06D010_F7D4"]
}

scene_F7D4_items = stac_client.search(
    collections=collections_to_search,
    datetime=f"{start_date}/{end_date}",
    intersects=aoi_bbox.boundary(),
    filter=scene_F7D4_filter,
).item_collection()

print(f"Found {len(scene_F7D4_items)} items")

### Advanced load

While `odc.stac` implements the option to group by `"solar_day"` it is also possible to group by an arbitrary metadata property. 
The below example groups items by scene, rather than by solar day.

In [ ]:
ds_by_scene = load(
    items=items,
    bands=assets_to_load,
    intersects=aoi_bbox.boundary(),
    crs=output_crs,
    resolution=output_res,
    groupby="sarard:scene_id",
    chunks={},
)

The effect of grouping by scene is that there are now 106 time-steps, compared to the 58 produced when grouping by solar day.

In [ ]:
ds_by_scene

The impact of grouping by scene is also visible when displaying the data.
There are two scenes for our area of interest that were captured on the same day. 
These are now displayed in separate time-steps.

In [ ]:
ds_by_scene.isel(time=range(0,3)).VV_gamma0.plot.imshow(col="time", col_wrap=3, robust=True, cmap="Greys_r")